In [38]:
import os
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.messages import HumanMessage, SystemMessage

# Middleware
- Tracking agent behavior with logging, analytics, and debugging
- Transforming prompts, tool selections, and output formatting
- Adding retries, fallbacks, and early termination logic
- Applying rate limits, guardrails, and PII detection

In [2]:
load_dotenv(override=True)

True

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Summarization
Automatically summarizes conversation history when approaching token limits, preserving recent messages while complressing older context.

Useful for:
- Long-running conversations that exceed context windows
- Multi-turn dialogues with extensive history
- Applications where preserving full conversation context matters


In [14]:
agent = create_agent(
    model = "gpt-5-mini",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "gpt-5-mini",
            trigger = ("messages", 10),
            keep = ("messages", 4)
        )
    ]
)

In [15]:
# thread_id to track user / thread
config = {
    "configurable": {"thread_id": "test-1"}
}

In [16]:
questions = [
    "What is 2+2?",
    "What is 5*5",
    "What is 10/4?",
    "What is 15-7?",
    "What is 4*4?",
    "What is 20+5?",
    "What is 99*99?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='0f15fc78-d1b5-4873-a121-ca39d7477321'), AIMessage(content='2 + 2 = 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 13, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EBkOnXpR3j4qGqPMfuCazMDqBsJqE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ff1d4-a897-7103-93e2-7cbdc8e410cf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 17, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_tok

In [20]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wi-fi
    """

agent = create_agent(
    model="gpt-5-mini",
    tools=[search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "gpt-5-mini",
            trigger = ("tokens", 550),
            keep = ("tokens", 200)
        )
    ]
)

config = {
    "configurable": {"thread_id": "test-2"}
}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [21]:
cities = ["Paris", "London", "Tokyo", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens}, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~191, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='3ffc26fd-ccd1-4880-9060-c2fc4eed1af5'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 134, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EBkam4VVugLOMoIsoJCnqSzpxZZPc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ff1df-fe7c-7353-a90c-4e7b3acaad71-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_ggyIA2Bhl6l4AY6PyfCCzDwn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

# Human in the loop

Pauses agent execution for human approval, editing, or rejection of tool calls before they execute.  Human in the loop is useful for:
- High stakes operations requiring human approval (e.g database writes, financial transaction)
- Compliance workflows where human oversight is mandatory
- Long running conversations where human feedback guides the agent

In [23]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send and email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [32]:
agent = create_agent(
    model="gpt-5-mini",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [33]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1 - request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with the suject 'Hello', and the body 'How are you?'")]},
    config=config
)

In [34]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with the suject 'Hello', and the body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e74b74bc-ce8c-4a0b-8bda-31c2b8ec5772'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 181, 'total_tokens': 346, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EBljofMBMpcyli1N5IvujwmSeRUEt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ff223-344c-7653-a9cf-971be201324e-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you

In [39]:
# Step 2 - approve
if "__interrupt__" in result:
    print("⏸️ Paused, needs approval.")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused, needs approval.
✅ Result: Done — I sent the email to john@test.com with subject "Hello" and body "How are you?".


In [40]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with the suject 'Hello', and the body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e74b74bc-ce8c-4a0b-8bda-31c2b8ec5772'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 181, 'total_tokens': 346, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EBljofMBMpcyli1N5IvujwmSeRUEt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ff223-344c-7653-a9cf-971be201324e-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you

In [41]:
# Reject
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1 - request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with the suject 'Hello', and the body 'How are you?'")]},
    config=config
)

In [ ]:
# Reject
if "__interrupt__" in result:
    print("⏸️ Paused, needs approval.")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"❌ Result: {result['messages'][-1].content}")

⏸️ Paused, needs approval.
❌ Result: I wasn’t able to send the message — the send-email tool was rejected, so the email was not sent.

Here’s what I attempted to send:
- To: john@test.com
- Subject: Hello
- Body: How are you?

Would you like me to try sending it again, edit the subject/body/recipient first, or cancel?


In [43]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with the suject 'Hello', and the body 'How are you?'", additional_kwargs={}, response_metadata={}, id='bbbeefb1-81f0-4d2b-8dfc-19fb2eade358'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 181, 'total_tokens': 282, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EBlrKHdwpeRwyfQDhldRdbHESdZyB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ff22a-4ef8-7e51-ae54-2d9ba9f8f926-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?

In [44]:
# Edit
config = {"configurable": {"thread_id": "test-edit"}}
# Step 1 - request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@test.com with the suject 'Hello', and the body 'How are you?'")]},
    config=config
)

In [48]:
# Edit
if "__interrupt__" in result:
    print("⏸️ Paused, needs approval.")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by a human before sending."
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    print(f"⏯️ Result: {result}")

⏸️ Paused, needs approval.
⏯️ Result: {'messages': [HumanMessage(content="Send email to wrong@test.com with the suject 'Hello', and the body 'How are you?'", additional_kwargs={}, response_metadata={}, id='3ba6923a-f9da-47ba-839f-7e643d0009ce'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 181, 'total_tokens': 282, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EBltlXQuhuVOSfvg5yGnKHnjPGDGF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ff22c-9f6c-75f1-8378-518eb8a1c35e-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 

In [47]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com with the suject 'Hello', and the body 'How are you?'", additional_kwargs={}, response_metadata={}, id='3ba6923a-f9da-47ba-839f-7e643d0009ce'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 181, 'total_tokens': 282, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EBltlXQuhuVOSfvg5yGnKHnjPGDGF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ff22c-9f6c-75f1-8378-518eb8a1c35e-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Cor